# 11-зертханалық жұмыс. Python арқылы ықтималдықтар мен статистикалық сипаттамалар

**Пән:** Деректерді интеллектуалды талдау  
**Ұзақтығы:** 2 академиялық сағат  
**Алдын ала дайындық:** 11-дәріс; NumPy, pandas, CSV оқу, негізгі графиктер.

## Оқу мақсаттары

- Толық берілген үлестірімдердің сипаттамаларын есептеу
- Аналитикалық есептеу мен симуляцияны салыстыру
- football.csv бойынша сипаттамалық есеп дайындау

## Ноутбукты іске қосу

Python ортасында мына пакеттерді орнатыңыз: `numpy`, `pandas`, `scipy`. Jupyter бағдарламасында Python 3 ядросын таңдап, ұяшықтарды жоғарыдан төмен орындаңыз. Соңғы код ұяшығында тапсырмаларды өз бетіңізше орындаңыз. Нәтижелерді зертханалық жұмыста көрсетілген `work_results` жолына сақтаңыз.

Оқулықтың архивін ашып, `notebooks/` және `data/` қалталарын бірге сақтаңыз. Ноутбукты бөлек жүктесеңіз, Jupyter немесе Colab файлдар панелінде ағымдағы жұмыс қалтасында `data` ішкі қалтасын жасап, оқулықтан алынған `football.csv` файлын сонда орналастырыңыз. Бірінші ұяшық дерек жолын тексереді.


In [ ]:
from pathlib import Path
import os

# Жұмыс қалтасын анықтау.
project_dir = Path.cwd()
if project_dir.name == "notebooks" and (
    (project_dir.parent / "data" / "football.csv").is_file()
    or (project_dir.parent / "index.html").is_file()
):
    project_dir = project_dir.parent

if not (project_dir / "data" / "football.csv").is_file():
    raise FileNotFoundError(
        "data/football.csv табылмады. Оқулықтан football.csv файлын жүктеп алып, "
        "ағымдағы жұмыс қалтасындағы data ішкі қалтасына орналастырыңыз."
    )

if Path.cwd() != project_dir:
    os.chdir(project_dir)

print("Жұмыс қалтасы:", project_dir)


## Есептің қойылымы және материалдар

Сіз нысандардың үш түрімен жұмыс істейсіз: ақырлы ықтималдық моделі, симуляцияланған тәжірибелер және футболшылардың тарихи файлы. Мақсат — нәтиже қай жерде шарттардан дәл есептелетінін, қай жерде кездейсоқ баға болатынын, ал қай жерде тек берілген жолдарды сипаттайтынын көрсету. Нәтижелерді бір орындалатын notebook не скрипт және қысқа есеп түрінде рәсімдеңіз.

Бастапқы 11-зертханалық жұмыстарда курстар аудиториясы, шығындар, шамаларды түрлендіру және қалыпты үлестірім туралы есептер бар. Бастапқы материалдардағы бірнеше кесте мен формула мәтінге айналдырылған нұсқада жоқ суреттермен берілген. Мұндай шарттар мұнда толық сандық деректермен ауыстырылды; бастапқы файлдың жауаптары жетіспейтін үлестірімдерді болжау үшін қолданылмайды. Білім беру платформасы туралы есеп таңдау тәсілі нақты анықталған күйінде сақталды.

Құралдар: Python, NumPy, pandas, SciPy және графикке арналған matplotlib. Жұмыс кезінде интернет қажет емес. Жергілікті data/football.csv қолданылады; өзге деректер төменде толық берілген. Жинақтың түбірінен іске қосыңыз. Барлық жасалатын файлдарды work_results қалтасына орналастырыңыз. Әрбір жеке симуляция үшін басқарылмайтын жаһандық генератордың орнына np.random.default_rng(42) қолданыңыз.

## Дайындық және талданған мысал

Мысал өзіндік тапсырмалардан өзгеше. X шамасы 0, 1, 2 мәндерін 0.25, 0.5, 0.25 ықтималдықтарымен қабылдасын. Онда E(X)=1, Var(X)=0.5. Y=10X+5 үшін күтім 15, дисперсия 50, стандартты ауытқу sqrt(50). Код оларды таңдамалық формуламен емес, тікелей анықтамасы бойынша есептейді.



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

out = Path('work_results')
out.mkdir(exist_ok=True)
x = np.array([0., 1., 2.])
p = np.array([0.25, 0.5, 0.25])
assert np.all(p >= 0) and np.isclose(p.sum(), 1)
mu = np.sum(x * p)
var = np.sum((x - mu) ** 2 * p)
y = 10 * x + 5
print(mu, var, np.sum(y * p))
print(np.sum((y - np.sum(y * p)) ** 2 * p))



1.0, 0.5, 15.0 және 50.0 аламыз. 0, 1, 2 мәндеріне жай x.mean() қолданып, ықтималдықтарды елемеуге болмайды: кейде орташа мәннің кездейсоқ сәйкес келуі қатені жасырады. Мұнда дисперсия үшін берілген үлестірім қолданылады, сондықтан ddof қажет емес. Ал бақыланған мәндерге арналған есептерде сипаттамалық немесе таңдамалық нұсқаны нақты таңдау керек.

Бастамас бұрын жергілікті файлдың және қажетті баған атауларының барын тексеріңіз:



In [ ]:
football = pd.read_csv(Path('data') / 'football.csv')
required = ['Age', 'Club', 'Position', 'SprintSpeed', 'Stamina']
assert set(required).issubset(football.columns)
assert football.shape == (12897, 42)
assert not football.isna().any().any()
print(football.loc[:, required].head(3))



Файл тарихи деректерден тұрады. Бас жиынтық пен жолдардың енгізілу механизмі белгісіз. Name бірегейлікке кепілдік бермейді, Unnamed: 0 — ескі индекс. Wage және Value бірліктері мен кезеңі расталмаған. Penalties — соғылған пенальти саны емес, дағды рейтингі. Бұл өрістерді оқиғалар санағышы немесе расталған ақша өлшемдері ретінде қолданбаңыз.

## Тапсырмалар

1. **Курсты немесе студентті таңдау.** Платформада әрқайсысында 25 адамнан 16 курс, 100 адамнан 3 курс және 300 адамы бар 1 курс бар. Әр адам тек бір курстан өтеді. Курстар мен студенттер санын табыңыз, содан кейін аудитория көлемі X-тің екі үлестірімін құрыңыз: курсты тең ықтималдықпен және студентті тең ықтималдықпен таңдағанда. size, p_course, p_student бағандары бар кесте ұсыныңыз. Әр баған ықтималдықтарының қосындысын тексеріп, екі күтімді есептеңіз. Әртүрлі жауаптардың неліктен қате емес екенін түсіндіріңіз. Студент таңдауды үш көлемнің бірін таңдаумен алмастырмаңыз.

2. **Толық дискретті үлестірім.** Сәтті сұраулар саны X 0, 1, 2, 3 мәндерін 0.1, 0.2, 0.4, 0.3 ықтималдықтарымен қабылдасын. E(X), E(X**2), Var(X), SD(X), P(X>=2) есептеңіз. Var(X)=E(X**2)-E(X)**2 формуласы мен ауытқулар квадраттарының тікелей қосындысының теңдігін тексеріңіз. Ықтималдықтарды тексеретін шағын функция жазыңыз: массив ұзындықтары бірдей, мәндер теріс емес, қосындысы бірге жақын болуы керек. Оны дұрыс емес ықтималдықтар жиынында тексеріп, табылған мәселе туралы хабарды есепке сақтаңыз. Notebook орындалуы үзілмеуі үшін әдейі туғызылған ерекше жағдайды өңдеңіз.

3. **Бірліктерді түрлендіру және бірлескен өзгергіштік.** Әр сәтті сұрау үшін 200 шартты бірлік есептеледі, тұрақты шығын 300. Пайда Y=200X-300. Күтім мен стандартты ауытқуды екі тәсілмен есептеңіз: Y мәндерінен тікелей және сызықтық түрлендіру қасиеттері арқылы. Содан кейін тәуелсіз Z шамасын беріңіз: 0, 2 мәндері, ықтималдықтары 0.6, 0.4. S=X+Z үшін күтім мен дисперсияны табыңыз. T=X+X жағдайын бөлек қарастырыңыз. T дисперсиясы X-тің екі тәуелсіз көшірмесінің дисперсиялары қосындысына неге тең емес екенін түсіндіріңіз.

4. **Қалыпты үлестірімнің құйрықтары мен квантильдері.** Оқу моделінде X балы mu=700 және sigma=120 параметрлерімен қалыпты үлестірілген. P(X>820), P(730<X<=820), q_0.90 квантилін және q_0.05-тен q_0.95-ке дейінгі аралықты табыңыз. Бірінші тармақ үшін sf, аралық үшін cdf айырмасын қолданыңыз. Құйрық бағыты көрінуі үшін математикалық оқиғаны және функция шақыруын қатар жазыңыз. Ықтималдықты есептеуге дейін z мәнін дөңгелектемеңіз. Модельдің анықталу облысы шектелмегенін, ал нақты балдардың шекаралары болуы мүмкін екенін көрсетіңіз.

5. **Ақырлы модельді симуляциялау.** 2-тапсырмадағы X үшін берілген p арқылы rng.choice пайдаланып, 10000 тәуелсіз нәтиже генерациялаңыз. Жиіліктерді ықтималдықтармен, таңдамалық орташа мәнді E(X)-пен салыстырыңыз. value, probability, relative_frequency, absolute_error кестесін құрыңыз. Жиіліктер қосындысын және ең үлкен абсолюттік айырманы есептеңіз. Жиіліктердің неліктен p-мен дәл сәйкес келуге міндетті емес екенін және бір seed модель сенімділігін тексермейтінін түсіндіріңіз.

6. **Қайталама таңдамалардағы орташа мән.** scale=3 экспоненциалдық үлестірімінен n=5 және n=80 көлемдері үшін 2000-нан орташа мән модельдеңіз. Әр n үшін орташа мәндердің орташа мәні мен стандартты ауытқуын бағалаңыз; соңғысын 3/sqrt(n) шамасымен салыстырыңыз. Бір суретте түсінікті белгілері бар екі орташа мән гистограммасын көрсетіңіз. Тығыздықтарды баған ықтималдықтарымен шатастырмаңыз. Екі бөлек қорытынды тұжырымдаңыз: шашырау және орташа мәндер үлестірімінің пішіні туралы. ОШТ бастапқы экспоненциалдық бақылаулардың пішінін өзгертпейтінін түсіндіріңіз.

7. **Нақты файлды сипаттау.** Age, SprintSpeed және Stamina үшін n, mean, median, ddof=1 болатын sd, Q1, Q3 және IQR есептеңіз. Age үшін ddof=0 және ddof=1 кезіндегі std мәндерін бөлек салыстырыңыз. Club бойынша сан мен орташа Age табыңыз, содан кейін ойыншылар бойынша орташа мәнді клубтық орташа мәндердің қарапайым ортасымен салыстырыңыз. Топ көлемдерінің бірдей болмауының әсерін түсіндіріңіз; сұрақты тұжырымдамай тұрып тәсілдердің бірі әрқашан «дұрысырақ» деген қорытынды жасамаңыз. Әр баға үшін талдау бірлігін көрсетіңіз.

8. **Қорытынды және өзіндік тексеру.** Ақырлы модель нәтижелері мен football сипаттамасын CSV-ге сақтаңыз. Есепте шарттардан дәл алынған тұжырымдарды, симуляция нәтижелерін және файлдың сипаттамалық көрсеткіштерін ажыратыңыз. Барлық ойыншыға жалпылаудың кемінде үш шектеуін келтіріңіз: белгісіз іріктеу, тарихи кезең, клуб ішіндегі ықтимал тәуелділік. Notebook таза іске қосылғанда жоғарыдан төменге дейін орындалатынын және бұрынғы әрекеттер айнымалыларына тәуелді еместігін тексеріңіз.

## Не тапсыру керек және қалай тексеру керек

Notebook немесе .py, түсіндірмелері бар қысқа есеп және work_results файлдарын тапсырыңыз. Нәтиже файлдарының атаулары: lab11_courses.csv, lab11_distribution.csv, lab11_football_summary.csv, lab11_means.png. Есепті notebook ішіндегі Markdown ұяшықтарына енгізуге болады. Скрипт үшін бөлек мәтіндік есеп қосыңыз. Суретті жеке үлкен есеп ретінде рәсімдеу талап етілмейді.

| Файл | Құрылымы | Тексеру |
|---|---|---|
| lab11_courses.csv | size, p_course, p_student | 3 жол, p бағандарының екі қосындысы да 1-ге тең |
| lab11_distribution.csv | value, probability, relative_frequency, absolute_error | 4 жол, алғашқы екі үлестірім қосындысы 1-ге тең |
| lab11_football_summary.csv | feature, n, mean, median, sd, q25, q75, iqr | 3 жол; iqr=q75-q25 |
| lab11_means.png | Екі таңдама көлемінің гистограммалары | Осьтер, түсіндірме және n мәні бар |

Ықтималдықтар мен жиіліктерді пайыз таңбасы бар жолдар емес, 0-ден 1-ге дейінгі сандар ретінде сақтаңыз. float мәндерін салыстыру үшін дәл теңдікті емес, негізделген рұқсатпен np.isclose қолданыңыз. CSV сандарын мерзімінен бұрын дөңгелектемей сақтаңыз. Есепте мағынаны жасырмаса, 3-4 таңбаға дейін дөңгелектеуге болады.

Қосымша тапсырма: football ішіндегі екі тәуелді дағды үшін ковариацияны есептеп, бірдей ddof=0 кезінде Var(A+B)=Var(A)+Var(B)+2Cov(A,B) теңдігін тексеріңіз. Бақыланған байланыс себепті байланыс емес екенін көрсетіңіз. Бұл тапсырма негізгі тармақтарды алмастырмайды және гипотезаларды тексеруді талап етпейді.

## Қорғауға арналған сұрақтар

1. Кездейсоқ студент курсының көлемі неліктен басқаша үлестірілген?
2. Неліктен Var(2X) шамасы 2Var(X)-ке тең емес?
3. sf пен cdf айырмасы қандай және оң жақ құйрық қашан маңызды?
4. Сіздің симуляцияңызда бақылаулардың SD шамасы мен орташалардың SD шамасы қалай ерекшеленеді?
5. Іріктеу туралы ақпаратсыз football бойынша қандай қорытынды жасауға болмайды?

Бағалау, барлығы 100 балл:

- 15 балл — курстардың екі үлестірімі және салмақтарды түсіндіру.
- 20 балл — күтім, дисперсия және тексерулері бар түрлендірулер.
- 15 балл — қалыпты ықтималдықтар мен квантильдер.
- 20 балл — қайталанатын симуляция, салыстыру және график.
- 20 балл — football сипаттамалық кестесі және шектеулер.
- 10 балл — қайталанатын іске қосу, файлдар және ауызша қорғау.

Құжаттама:

- [NumPy: choice](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html).
- [SciPy: norm](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html).
- [pandas: топтау](https://pandas.pydata.org/docs/user_guide/groupby.html).

## Өзіндік жұмыс

Осы зертханалық жұмыстың тапсырмаларын төмендегі ұяшықта орындаңыз. Қажет болса, код пен түсіндірме үшін қосымша ұяшықтар қосыңыз. Есептеулерді аяқтаған соң ядроны қайта іске қосып, ұяшықтарды басынан бастап орындаңыз.

In [ ]:
# Тапсырмаларды орындау кодын осы жерге жазыңыз.
